In [1]:
import json
import numpy as np
import pandas as pd
from collections import Counter

# ── RUTAS ─────────────────────────────────────────────────────────────────────
VAL_DATA_PATH  = "../data/last_task/val.json"       # tu JSON de validación
PREDS_PATH     = "../data/last_task/xlm-roberta-base-76.7.json"   # el JSON que guarda evaluate()

# ── CARGA ─────────────────────────────────────────────────────────────────────
with open(VAL_DATA_PATH,  "r") as f: val_data  = json.load(f)
with open(PREDS_PATH,     "r") as f: preds_raw = json.load(f)

preds = preds_raw["predictions"]
assert len(preds) == len(val_data), f"Mismatch: {len(preds)} preds vs {len(val_data)} samples"

# ── CONSTRUCCIÓN DEL DATAFRAME ────────────────────────────────────────────────
rows = []
for sample, pred in zip(val_data, preds):
    votes     = sample["label"] if isinstance(sample["label"], list) else [sample["label"]]
    n_votes   = len(votes)
    n_pos     = sum(votes)
    certainty = abs(n_pos / n_votes - 0.5) * 2   # 0 = total desacuerdo, 1 = acuerdo total

    has_ocr   = bool(sample.get("qwen_ocr",          "").strip())
    has_trans = bool(sample.get("qwen_transcription", "").strip())
    has_reas  = bool(sample.get("qwen_reasoning",     "").strip())

    physio     = sample.get("physio", {})
    n_eeg      = len(physio.get("EEG", []))
    n_et       = len(physio.get("ET",  []))
    n_hr       = len(physio.get("HR",  []))

    rows.append({
        "id":          sample["id_Tiktok"],
        "label":       pred["label"],
        "pred":        pred["pred"],
        "prob":        pred["prob"],              # prob de clase 1
        "correct":     pred["label"] == pred["pred"],
        "certainty":   round(certainty, 3),
        "n_annotators": n_votes,
        "has_ocr":     has_ocr,
        "has_trans":   has_trans,
        "has_reas":    has_reas,
        "n_modalities": int(has_ocr) + int(has_trans) + int(has_reas),
        "n_eeg":       n_eeg,
        "n_et":        n_et,
        "n_hr":        n_hr,
        "has_physio":  n_eeg > 0 or n_et > 0,
        "ocr_len":     len((sample.get("qwen_ocr")          or "").split()),
        "trans_len":   len((sample.get("qwen_transcription") or "").split()),
        "reas_len":    len((sample.get("qwen_reasoning")     or "").split()),
    })

df = pd.DataFrame(rows)

# ─────────────────────────────────────────────────────────────────────────────
# 1. RESUMEN GLOBAL
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("=" * 60)
print("1. RESUMEN GLOBAL")
print("=" * 60)
print(classification_report(df["label"], df["pred"], target_names=["No sarcasmo", "Sarcasmo"]))

cm = confusion_matrix(df["label"], df["pred"])
cm_df = pd.DataFrame(cm,
    index   = ["Real: No sarcasmo", "Real: Sarcasmo"],
    columns = ["Pred: No sarcasmo", "Pred: Sarcasmo"]
)
print("\nMatriz de confusión:")
print(cm_df.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 2. ERRORES POR CERTEZA DEL ANNOTATOR
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. ACCURACY POR CERTEZA DEL ANNOTATOR")
print("=" * 60)

df["certainty_bin"] = pd.cut(df["certainty"], bins=[0, 0.25, 0.5, 0.75, 1.01],
                              labels=["0-25% (muy ambiguo)", "25-50%", "50-75%", "75-100% (claro)"])
cert_table = df.groupby("certainty_bin", observed=True).agg(
    n          = ("correct", "count"),
    accuracy   = ("correct", "mean"),
    f1_macro   = ("correct", lambda x: f1_score(
                    df.loc[x.index, "label"],
                    df.loc[x.index, "pred"], average="macro", zero_division=0)),
    prob_media = ("prob",    "mean"),
).round(3)
print(cert_table.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 3. ERRORES POR MODALIDAD TEXTUAL DISPONIBLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. ACCURACY POR MODALIDADES TEXTO DISPONIBLES")
print("=" * 60)

for col, name in [("has_ocr", "OCR"), ("has_trans", "Transcripción"), ("has_reas", "Reasoning")]:
    tbl = df.groupby(col).agg(
        n        = ("correct", "count"),
        accuracy = ("correct", "mean"),
    ).round(3)
    tbl.index = tbl.index.map({
        False: f"Sin {name}",
        True:  f"Con {name}"
    })
    print(f"\n{name}:")
    print(tbl.to_string())

print("\nPor nº de modalidades texto disponibles:")
print(df.groupby("n_modalities").agg(
    n        = ("correct", "count"),
    accuracy = ("correct", "mean"),
).round(3).to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 4. ERRORES POR FISIOLOGÍA
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("4. ACCURACY CON/SIN FISIOLOGÍA")
print("=" * 60)

print(df.groupby("has_physio").agg(
    n        = ("correct", "count"),
    accuracy = ("correct", "mean"),
    f1_macro = ("correct", lambda x: f1_score(
                    df.loc[x.index, "label"],
                    df.loc[x.index, "pred"], average="macro", zero_division=0)),
).round(3).rename(index={True: "Con fisiología", False: "Sin fisiología"}).to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 5. DISTRIBUCIÓN DE PROBABILIDADES — DONDE EL MODELO DUDA
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("5. DISTRIBUCIÓN DE PROBABILIDADES")
print("=" * 60)

df["prob_bin"] = pd.cut(df["prob"], bins=[0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0],
                         labels=["0-30", "30-40", "40-50", "50-60", "60-70", "70-100"])
prob_table = df.groupby("prob_bin", observed=True).agg(
    n           = ("correct", "count"),
    n_correct   = ("correct", "sum"),
    accuracy    = ("correct", "mean"),
).round(3)
prob_table["n_correct"] = prob_table["n_correct"].astype(int)
print(prob_table.to_string())

uncertain = df[(df["prob"] > 0.4) & (df["prob"] < 0.6)]
print(f"\nMuestras en zona de duda (prob 0.4-0.6): {len(uncertain)} "
      f"({100*len(uncertain)/len(df):.1f}% del total)")
print(f"Accuracy en zona de duda: {uncertain['correct'].mean():.3f}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. PEORES PREDICCIONES — CASOS CONCRETOS PARA INSPECCIONAR
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("6. FALSOS POSITIVOS MÁS CONFIADOS (predijo sarcasmo, era no sarcasmo)")
print("=" * 60)

fp = df[(df["pred"] == 1) & (df["label"] == 0)].sort_values("prob", ascending=False).head(10)
print(fp[["id", "prob", "certainty", "has_ocr", "has_trans", "has_reas", "n_eeg"]].to_string(index=False))

print("\nFALSOS NEGATIVOS MÁS CONFIADOS (predijo no sarcasmo, era sarcasmo)")
fn = df[(df["pred"] == 0) & (df["label"] == 1)].sort_values("prob", ascending=True).head(10)
print(fn[["id", "prob", "certainty", "has_ocr", "has_trans", "has_reas", "n_eeg"]].to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# 7. LONGITUD DE TEXTO VS ERROR
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("7. LONGITUD DE TEXTO VS ACCURACY")
print("=" * 60)

for col, name in [("ocr_len", "OCR"), ("trans_len", "Transcripción"), ("reas_len", "Reasoning")]:
    df[f"{col}_bin"] = pd.qcut(df[col], q=3, labels=["corto", "medio", "largo"], duplicates="drop")
    tbl = df.groupby(f"{col}_bin", observed=True).agg(
        n        = ("correct", "count"),
        accuracy = ("correct", "mean"),
    ).round(3)
    print(f"\n{name} por longitud:")
    print(tbl.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 8. RESUMEN FINAL — QUÉ ESTÁ FALLANDO
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("8. RESUMEN: PATRONES DE ERROR")
print("=" * 60)

error_df = df[~df["correct"]]
print(f"Total errores: {len(error_df)} / {len(df)} ({100*len(error_df)/len(df):.1f}%)\n")

print("Distribución de errores por certeza del annotator:")
print(error_df["certainty_bin"].value_counts().sort_index().to_string())

print("\nDistribución de errores por nº modalidades texto:")
print(error_df["n_modalities"].value_counts().sort_index().to_string())

print("\nErrores con vs sin fisiología:")
print(error_df["has_physio"].value_counts().rename({True: "Con fisiología", False: "Sin fisiología"}).to_string())

print("\nThreshold óptimo para F1 macro:")
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.2, 0.8, 0.01):
    preds_t = (df["prob"] >= t).astype(int)
    f1      = f1_score(df["label"], preds_t, average="macro")
    if f1 > best_f1:
        best_f1, best_t = f1, t
print(f"  Threshold={best_t:.2f}  →  F1 macro={best_f1:.4f}  (actual con 0.5: "
      f"{f1_score(df['label'], df['pred'], average='macro'):.4f})")

# ─────────────────────────────────────────────────────────────────────────────
# 9. ANÁLISIS DETALLADO: CASOS EXTREMOS (CERTEZA 100% Y 0%)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("9. CASOS EXTREMOS: ERRORES CLAROS Y MÁXIMA AMBIGÜEDAD")
print("=" * 60)

# --- 9.1 ERRORES EN CASOS CLARÍSIMOS (Certeza 100%) ---
# Humanos están totalmente de acuerdo, pero el modelo falla.
errores_100_cert = df[(df["certainty"] == 1.0) & (~df["correct"])]

print(f"\n[!] Errores en casos CLAROS (Certeza 100%): {len(errores_100_cert)} muestras")

if not errores_100_cert.empty:
    # Separar en Falsos Positivos y Falsos Negativos
    fp_claros = errores_100_cert[errores_100_cert["pred"] == 1].sort_values("prob", ascending=False)
    fn_claros = errores_100_cert[errores_100_cert["pred"] == 0].sort_values("prob", ascending=True)

    print(f"\n  -> Falsos Positivos (Era NO sarcasmo indiscutible, pero predijo SARCASMO): {len(fp_claros)}")
    if not fp_claros.empty:
        print(fp_claros[["id", "prob", "n_modalities", "has_physio"]].head(10).to_string(index=False))

    print(f"\n  -> Falsos Negativos (Era SARCASMO indiscutible, pero predijo NO sarcasmo): {len(fn_claros)}")
    if not fn_claros.empty:
        print(fn_claros[["id", "prob", "n_modalities", "has_physio"]].head(10).to_string(index=False))

# --- 9.2 CASOS DE MÁXIMA AMBIGÜEDAD (Certeza 0%) ---
# Humanos están divididos (ej. 1 a favor, 1 en contra). 
casos_0_cert = df[df["certainty"] == 0.0]

print(f"\n[?] Casos de máxima AMBIGÜEDAD (Certeza 0%): {len(casos_0_cert)} muestras")

if not casos_0_cert.empty:
    print(f"  -> Accuracy del modelo en estos casos (prácticamente al azar): {casos_0_cert['correct'].mean():.3f}")
    
    # Ver casos ambiguos donde el modelo está "demasiado seguro" de sí mismo
    # Probabilidad > 0.8 (muy seguro de sarcasmo) o < 0.2 (muy seguro de no sarcasmo)
    seguros_en_ambiguos = casos_0_cert[(casos_0_cert["prob"] >= 0.8) | (casos_0_cert["prob"] <= 0.2)]
    
    print(f"\n  -> Casos donde los humanos dudan al máximo, pero el modelo está MUY SEGURO (prob >= 0.8 o <= 0.2): {len(seguros_en_ambiguos)}")
    if not seguros_en_ambiguos.empty:
        # Ordenamos por la 'distancia' al centro para ver los más extremos primero
        seguros_en_ambiguos = seguros_en_ambiguos.assign(extremo=abs(seguros_en_ambiguos["prob"] - 0.5))
        seguros_en_ambiguos = seguros_en_ambiguos.sort_values("extremo", ascending=False)
        print(seguros_en_ambiguos[["id", "label", "pred", "prob"]].head(10).to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# 10. SESGO DE GÉNERO Y ANÁLISIS DE TRUNCAMIENTO (ROBERTA LIMITS)
# ─────────────────────────────────────────────────────────────────────────────

# Expandimos la lista a inglés y términos comunes en estos datasets
gender_terms = [
    'woman', 'women', 'girl', 'her', 'she', 'ex', 'girlfriend', 'wife', 
    'mujer', 'mujeres', 'chica', 'ella', 'novia', 'esposa',
    'untrustworthy', 'lying', 'always', 'toxic', 'generalization'
]

def analyze_bias_and_length(row, tokenizer_estimate=1.3):
    # 1. Detección de términos
    reasoning = str(row.get('qwen_reasoning', '')).lower()
    has_gender = any(word in reasoning for word in gender_terms)
    
    # 2. Estimación de Tokens (XLM-R usa SentencePiece, aprox 1.3 tokens por palabra)
    # Si sumas todas las modalidades que le pasas al modelo:
    total_words = row['ocr_len'] + row['trans_len'] + row['reas_len']
    est_tokens = total_words * tokenizer_estimate
    
    return pd.Series([has_gender, est_tokens, est_tokens > 256])

df[['mention_gender', 'est_tokens', 'is_truncated']] = df.apply(analyze_bias_and_length, axis=1)

print("\n" + "=" * 60)
print("10. ANÁLISIS DE SESGO Y TRUNCAMIENTO (Límite 512 Tokens)")
print("=" * 60)

# --- Sub-análisis A: Género ---
print("\n[A] Rendimiento en casos con temática de Género/Relaciones:")
gender_stats = df.groupby('mention_gender').agg(
    n=('correct', 'count'),
    accuracy=('correct', 'mean'),
    avg_prob=('prob', 'mean')
).round(3)
print(gender_stats.rename(index={True: "Menciona Género", False: "Otros"}).to_string())

# --- Sub-análisis B: Truncamiento ---
print("\n[B] Rendimiento vs. Truncamiento (¿Se corta el texto?):")
trunc_stats = df.groupby('is_truncated').agg(
    n=('correct', 'count'),
    accuracy=('correct', 'mean'),
    avg_tokens=('est_tokens', 'mean')
).round(3)
print(trunc_stats.rename(index={True: "POSIBLE TRUNCADO (>512)", False: "Texto Completo"}).to_string())

# --- Sub-análisis C: El "Peor Escenario" ---
# Casos de género que además son largos (donde el modelo pierde el hilo)
bad_combo = df[(df['mention_gender'] == True) & (df['is_truncated'] == True)]
if not bad_combo.empty:
    print(f"\n[!] Casos críticos (Género + Truncados): {len(bad_combo)}")
    print(f"    Accuracy en este grupo: {bad_combo['correct'].mean():.3f}")

1. RESUMEN GLOBAL
              precision    recall  f1-score   support

 No sarcasmo       0.77      0.80      0.78       199
    Sarcasmo       0.77      0.73      0.75       180

    accuracy                           0.77       379
   macro avg       0.77      0.77      0.77       379
weighted avg       0.77      0.77      0.77       379


Matriz de confusión:
                   Pred: No sarcasmo  Pred: Sarcasmo
Real: No sarcasmo                160              39
Real: Sarcasmo                    49             131

2. ACCURACY POR CERTEZA DEL ANNOTATOR
                   n  accuracy  f1_macro  prob_media
certainty_bin                                       
25-50%           107     0.654     0.652       0.491
75-100% (claro)  272     0.812     0.809       0.453

3. ACCURACY POR MODALIDADES TEXTO DISPONIBLES

OCR:
           n  accuracy
has_ocr               
Sin OCR   64     0.719
Con OCR  315     0.778

Transcripción:
                     n  accuracy
has_trans                    